In [1]:
from pathlib import Path
import pandas as pd

project_dir = Path(r"C:\Users\HP-ZBOOK i7\graphrag_test\pilot_01")
csv_path = project_dir / "inspection" / "09_text_units_all.csv"

text_units = pd.read_csv(csv_path)

print("Number of chunks:", len(text_units))
print("Columns:", text_units.columns.tolist())

text_units[["human_readable_id", "n_tokens"]]

Number of chunks: 17
Columns: ['id', 'human_readable_id', 'text', 'n_tokens', 'document_id', 'entity_ids', 'relationship_ids', 'covariate_ids']


,human_readable_id,n_tokens
0,0,1200
1,1,1200
2,2,1200
3,3,1200
4,4,1200
5,5,1200
6,6,1200
7,7,1200
8,8,1200
9,9,1200


In [2]:
for i, row in text_units.iterrows():
    text = row["text"]

    print("=" * 80)
    print(f"CHUNK {i + 1}")
    print("Tokens:", row["n_tokens"])
    print("Beginning:", repr(text[:200]))
    print("Ending:", repr(text[-200:]))

CHUNK 1
Tokens: 1200
Beginning: 'Österreichische Photovoltaik-\nStrategie\n\nZielsetzungen und Aktionsfelder eines strategischen Ausbauprozesses\n\nsowie Maßnahmen für einen koordinierten Ausbau der Photovoltaik\n\nin Österreich\n\n\x0cImpressum'
Ending: '..................................................................... 32\n\n6.7  Aktionsfeld „Partizipation“ ............................................................................................ '
CHUNK 2
Tokens: 1200
Beginning: '\n\n6.4.1  Maßnahmen ......................................................................................................... 27\n\n6.5  Aktionsfeld „Heimische PV-Wirtschaft“ ............................'
Ending: ' CO₂-Emissionen eingeführt. Parallel dazu erfolgt eine\n\nRückvergütung über den sogenannten „Klimabonus“ an Haushalte. Zudem wurden die fi-\n\nnanziellen Mittel des BMK für die Transformation des Energie'
CHUNK 3
Tokens: 1200
Beginning: ' Im Jahr 2023 waren 6,3 TWh er-\n\nreicht.\n\nAb Okto

In [3]:
for i in range(len(text_units) - 1):
    current_text = text_units.loc[i, "text"]
    next_text = text_units.loc[i + 1, "text"]

    print("=" * 80)
    print(f"BOUNDARY: CHUNK {i + 1} → CHUNK {i + 2}")
    print()
    print("END OF CURRENT CHUNK:")
    print(current_text[-300:])
    print()
    print("START OF NEXT CHUNK:")
    print(next_text[:300])
    print()

BOUNDARY: CHUNK 1 → CHUNK 2

END OF CURRENT CHUNK:
.......................................... 32

6.6.1  Maßnahmen ......................................................................................................... 32

6.7  Aktionsfeld „Partizipation“ ............................................................................................ 

START OF NEXT CHUNK:


6.4.1  Maßnahmen ......................................................................................................... 27

6.5  Aktionsfeld „Heimische PV-Wirtschaft“ ........................................................................ 29

6.5.1  Maßnahmen ..................................

BOUNDARY: CHUNK 2 → CHUNK 3

END OF CURRENT CHUNK:
e schrittweise steigende CO₂-Bepreisung für nicht vom EU-Emis-

sionshandel umfasste fossil bedingte CO₂-Emissionen eingeführt. Parallel dazu erfolgt eine

Rückvergütung über den sogenannten „Klimabonus“ an Haushalte. Zudem wurden die fi-

nanziellen Mittel des BMK für die

## Stage 2 — Text-Unit and Chunking Quality Audit

### Objective

The purpose of this stage was to evaluate how GraphRAG divided the converted Austrian Photovoltaic Strategy into text units and whether the resulting chunk boundaries could negatively affect entity and relationship extraction.

The analysis used `09_text_units_all.csv`, which contains the exact text units processed by GraphRAG during graph extraction.

The chunking configuration used in Experiment 1 was:

```yaml
chunking:
  type: tokens
  size: 1200
  overlap: 100
  encoding_model: o200k_base
```

This means that GraphRAG divided the document according to token count rather than according to pages, paragraphs, headings, or semantic sections. Each full chunk was limited to 1,200 tokens, and approximately 100 tokens were repeated between consecutive chunks.

---

### Basic chunking results

GraphRAG produced **17 text units**:

| Chunk group | Number of chunks |                                               Token count |
| ----------- | ---------------: | --------------------------------------------------------: |
| Full chunks |               16 |                                         1,200 tokens each |
| Final chunk |                1 |                                                124 tokens |
| Total       |               17 | Approximately 19,324 tokens before accounting for overlap |

The fact that the first 16 chunks contain exactly 1,200 tokens confirms that the division was primarily determined by a fixed token limit rather than by the document’s logical structure.

---

### Method

The beginning and ending of every chunk were inspected. In addition, all 16 transitions between adjacent chunks were compared.

The audit focused on:

* Broken sentences
* Broken words
* Separation of headings from their explanations
* Separation of policy names from related measures
* Separation of numerical targets from dates or units
* Duplication caused by overlap
* Mixture of unrelated document sections
* Inclusion of headers, footers, references, and administrative material
* Potential effects on downstream entity and relationship extraction

---

## Overlap analysis

Every pair of consecutive chunks contained an exact repeated text segment.

| Boundary            | Exact repeated characters |
| ------------------- | ------------------------: |
| Chunk 1 → Chunk 2   |                       740 |
| Chunk 2 → Chunk 3   |                       367 |
| Chunk 3 → Chunk 4   |                       411 |
| Chunk 4 → Chunk 5   |                       379 |
| Chunk 5 → Chunk 6   |                       454 |
| Chunk 6 → Chunk 7   |                       384 |
| Chunk 7 → Chunk 8   |                       406 |
| Chunk 8 → Chunk 9   |                       373 |
| Chunk 9 → Chunk 10  |                       416 |
| Chunk 10 → Chunk 11 |                       384 |
| Chunk 11 → Chunk 12 |                       404 |
| Chunk 12 → Chunk 13 |                       405 |
| Chunk 13 → Chunk 14 |                       407 |
| Chunk 14 → Chunk 15 |                       460 |
| Chunk 15 → Chunk 16 |                       338 |
| Chunk 16 → Chunk 17 |                       335 |

The variation in character counts is expected because the configured overlap is measured in tokens, not characters.

### Positive effect of overlap

The 100-token overlap generally prevented complete loss of information at chunk boundaries. When a sentence or paragraph crossed a boundary, a substantial part of that context was repeated in the following chunk.

Therefore, there is no clear evidence that important text disappeared solely because it crossed from one chunk into another.

For example, the boundary between Chunks 6 and 7 occurs near a discussion of the `Elektrizitätswirtschaftsgesetz`. Although Chunk 6 ends during the law’s name, Chunk 7 begins earlier because of the overlap and repeats the surrounding discussion.

The overlap consequently reduced the risk that the LLM would receive only an isolated continuation without any preceding context.

---

## Identified chunking problems

### 1. Chunking is not aligned with the document’s semantic structure

The chunks were determined by a fixed token count. They were not aligned with:

* Chapters
* Subsections
* Action fields
* Individual measures
* Paragraphs
* Complete sentences
* Tables or lists
* Page boundaries

As a result, some chunks contain the end of one topic and the beginning of another.

This creates a risk that the LLM may associate entities appearing near each other because of chunk construction rather than because the source expresses a meaningful relationship between them.

---

### 2. Front matter and substantive content are mixed

Chunk 1 contains several document components:

* Title and cover material
* Publication information
* Foreword
* Table of contents
* Beginning of the substantive document

These components serve different purposes. Publication metadata, the table of contents, and substantive policy text should not necessarily receive equal treatment during domain-graph extraction.

Processing all of them together may encourage GraphRAG to extract:

* Publishers and contact information as central domain entities
* Table-of-contents headings as separate events
* Relationships based primarily on document layout
* Repeated versions of concepts appearing both in the table of contents and the body

---

### 3. Section boundaries are crossed

Several chunk boundaries occur within or near major document sections.

Examples include:

* The transition into the legal-framework action field
* The transition into energy infrastructure
* The transition into acceptance
* The transition into domestic PV industry
* The transition into participation
* The transition from substantive text to references and abbreviations

The overlap preserves some surrounding context, but the LLM still processes each chunk independently during extraction. A chunk may therefore contain only part of the larger section structure.

This can make it harder to determine whether a statement is:

* A general strategic vision
* A specific recommended measure
* Background information
* A legal requirement
* A forecast
* A target
* A bibliographic reference

---

### 4. Sentences and words are divided at boundaries

Some chunks begin or end in the middle of sentences or words.

Examples include:

```text
Elektrizitätswirtschaftsgesetz (El...
```

```text
...Eu-
```

followed later by:

```text
-ropäische
```

and a chunk beginning with:

```text
iewende
```

These problems arise from a combination of token-based chunking and the PDF-conversion artifacts identified in Stage 1.

The overlap usually retains surrounding context, but it does not repair malformed words. It can instead repeat the malformed text in multiple chunks.

---

### 5. Lists may lose their structure

Some boundaries occur near bullet-point lists describing benefits, action fields, or measures.

The converted text already contains imperfect bullet formatting, including repeated bullet symbols and broken lines. Token-based chunking can divide these lists further.

This may cause GraphRAG to:

* Treat list fragments as independent concepts
* Lose the relationship between the list and its heading
* Associate a benefit with the wrong preceding statement
* Extract vague entities from incomplete bullet points

---

### 6. Footnotes and references are mixed with substantive text

Several chunks include:

* Footnote numbers
* Source citations
* URLs
* Publication titles
* Page numbers
* Bibliographic references

For example, a chunk boundary near the energy-infrastructure section includes a study citation and URL immediately before the following action-field heading.

These elements are valuable for provenance but should not necessarily be treated as ordinary domain statements. Without structural differentiation, the extraction model may turn:

* Authors
* Publication titles
* Websites
* Citation fragments

into entities and relationships that are irrelevant to the intended solar-market KG.

---

### 7. Conversion noise is repeated through overlap

Stage 1 identified substantial PDF-conversion artifacts, including broken German words, excessive spacing, headers, footers, and page numbers.

Chunk overlap repeats some of these artifacts across consecutive chunks.

Therefore, overlap has two opposing effects:

* It preserves useful context.
* It also duplicates noise.

This duplication may contribute to repeated extraction, longer entity descriptions, unnecessary relationships, and additional LLM cost.

---

## Detailed finding: Chunk 17

Chunk 17 is especially problematic.

It contains only **124 tokens** and consists mainly of:

* The end of the abbreviation list
* Repeated document titles
* Page numbers
* Ministry contact information
* A telephone number
* An email address
* A website address

Its beginning is:

```text
att

Terawattstunde

Umweltverträglichkeitsprüfung
```

The initial fragment `att` is the end of the word `Terawatt`, demonstrating that the final chunk begins in the middle of a word.

Chunk 17 contains approximately 386 characters. Of these, 335 characters exactly repeat the ending of Chunk 16. It therefore adds only approximately **51 new characters**.

This means that most of Chunk 17 is redundant.

### Downstream impact of Chunk 17

Despite containing little substantive solar-market information, Chunk 17 is linked through `text_unit_ids` to:

* **9 entities**
* **13 relationships**

Examples of linked entities include:

* `SERVICEBUERO@BMK.GV.AT`
* `BMK.GV.AT`
* `RADETZKYSTRASSE 2, 1030 WIEN`
* `TERAWATTSTUNDE`
* `UMWELTVERTRÄGLICHKEITSPRÜFUNG`

Some of these are factually identifiable but irrelevant to the intended KG. For example, the ministry’s email address and website do not help explain relationships among Austrian solar-market policy, market, and technical factors.

Some entities also received unsuitable types. For example:

```text
UMWELTVERTRÄGLICHKEITSPRÜFUNG → EVENT
```

This is better understood as a regulatory or procedural concept, not an event.

A relationship linked to this material states that a terawatt-hour is “likely” a measurement related to the strategy’s goals or achievements. The word `likely` indicates that GraphRAG introduced an inference rather than extracting an explicit relationship from the abbreviation list.

Chunk 17 therefore demonstrates how administrative back matter can produce irrelevant entities, incorrect entity types, and speculative relationships.

---

## Separation of chunking errors from conversion errors

Not every problem visible at a chunk boundary was caused by chunking.

The following problems primarily originate in PDF conversion:

* Broken German words
* Repeated headers and footers
* Excessive spacing
* Footnote placement
* Imperfect reading order
* URLs inserted into the main text
* Page numbers mixed with sentences

The following problems primarily originate in chunking:

* Fixed-length rather than section-aware division
* Mixing multiple document functions in one chunk
* Processing low-value back matter as ordinary domain text
* Repeating noisy content through overlap
* Creating a nearly redundant final chunk
* Splitting sections, lists, and sentences at arbitrary token positions

Some errors arise through interaction between the two stages. For example, a word first broken during PDF conversion can then be repeated across chunks because of overlap.

---

## Overall assessment

The 100-token overlap worked as intended in the narrow sense that it preserved context around chunk boundaries. There is no strong evidence of major textual information loss caused solely by chunk division.

However, the fixed 1,200-token strategy does not respect the semantic and visual structure of the Austrian Photovoltaic Strategy. It processes front matter, substantive policy content, references, abbreviations, and contact information using the same extraction procedure.

As a result, chunking contributes to:

* Noisy extraction
* Irrelevant entities
* Repeated evidence
* Potential duplicate descriptions
* Speculative relationships
* Loss of distinction between document sections
* Unnecessary processing cost

### Severity assessment

**Chunking quality: Moderate concern**

Chunking is not currently the most severe suspected bottleneck because overlap substantially protects against context loss. Nevertheless, section-insensitive chunking and the inclusion of low-value document regions clearly reduce graph precision.

Entity typing and schema limitations are expected to be more severe, but this must be confirmed in the next audit stage.

---

## Recommendations for Experiment 2

The Experiment 1 text units and outputs should remain unchanged as the baseline.

A future Experiment 2 should consider the following changes:

### 1. Exclude or separately process low-value document regions

Potentially exclude or separately label:

* Cover pages
* Imprint
* Table of contents
* Repeated headers and footers
* Bibliography
* Abbreviation list
* Blank pages
* Contact-information page

These regions may still be preserved for provenance, but they should not automatically receive the same entity-relationship extraction treatment as substantive policy sections.

### 2. Use section-aware preprocessing

Preserve headings such as:

```text
6.1 Aktionsfeld „Rechtlicher Rahmen“
6.2 Aktionsfeld „Energieinfrastrukturen“
6.3 Aktionsfeld „Wirtschaftlicher Photovoltaik-Anlagenbetrieb“
```

Each heading should remain connected to its corresponding explanatory text and measures.

### 3. Add metadata to chunks

Where supported, each chunk should contain metadata such as:

* Document ID
* Document title
* Page number or page range
* Section heading
* Source language
* Publisher
* Publication year

This would improve provenance and help distinguish substantive content from references or administrative material.

### 4. Test a structure-aware chunking strategy

A future experiment could compare:

* Current fixed 1,200-token chunks
* Sentence-aware chunks
* Section-aware chunks with a token limit

The purpose would not be to identify one universally optimal chunk size, but to determine which approach better preserves complete policy measures and reduces irrelevant extraction.

### 5. Retain some overlap

Overlap should not necessarily be removed. It protected against context loss in Experiment 1.

However, the overlap should be applied after document cleaning and structural segmentation so that it repeats meaningful content rather than headers, references, and contact information.

### 6. Filter extremely small and redundant final chunks

A chunk that consists almost entirely of overlap and administrative material should not automatically be sent for domain entity and relationship extraction.

A minimum-new-content rule or back-matter filter could prevent cases such as Chunk 17.

---

## Stage 2 conclusion

The Austrian Photovoltaic Strategy was divided into 17 token-based text units. Sixteen chunks reached the configured maximum of 1,200 tokens, while the final chunk contained 124 tokens.

The 100-token overlap successfully preserved substantial context across all 16 boundaries, and no major content loss caused solely by chunk division was identified. Nevertheless, the chunks were not aligned with the document’s sections, sentences, policy measures, lists, or document functions.

Fixed-length chunking repeated PDF-conversion noise, mixed substantive content with references and administrative material, and produced a nearly redundant final chunk. Chunk 17 provides a concrete example: it consists mainly of abbreviations and contact details but is linked to nine entities and thirteen relationships, including irrelevant and speculative graph content.

The evidence indicates that chunking is a **moderate bottleneck**. It should be improved in Experiment 2 through document-region filtering, section-aware processing, provenance metadata, and careful retention of overlap. However, the baseline must remain unchanged until the entity, relationship, and community-report audits are complete.

